# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nipun-Wanjale-dev/ML-Internship-NSW/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will rank content pages using a simple opportunity score based on search visibility and CTR.

The rule prioritizes pages that have meaningful Google Search Console impression volume but relatively low CTR. The idea is that a page already receiving search visibility may have an opportunity to improve clicks without requiring a completely new content asset.

The rule uses two signals:

- **Impressions:** higher impressions indicate greater potential reach.
- **CTR:** lower CTR indicates a possible click-through opportunity.

The baseline has one reason code:

- `HIGH_VOLUME_LOW_CTR` — the page has relatively high search impressions and relatively low CTR.

The action is:

- `REVIEW_CTR` — review the page's title/snippet/search intent alignment.

This is a deliberately simple baseline. It is intended as a decision-support queue, not proof that changing a page will increase traffic.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

print("Imports successful")

Imports successful


In [2]:
# Safe Colab authentication.
# The token must be stored in Colab Secrets as HF_TOKEN.
# Never paste the token directly into this notebook.

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add a READ Hugging Face token "
        "to Colab Secrets with the name HF_TOKEN."
    )

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB + Hugging Face connection ready.")

DuckDB + Hugging Face connection ready.


In [3]:
MARCH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(MARCH)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
# First inspect the schema.
schema = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{MARCH}')
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [5]:
# Load March data at content level.
#
# The warehouse grain is:
# report_date × client_hash_id × content_hash_id
#
# For this baseline we aggregate the March daily observations
# to one row per client × content item.

march = con.sql(f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet('{MARCH}')
    WHERE gsc_data_available IS TRUE
),

content_month AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                SUM(gsc_clicks) * 100.0
                / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_pct,

        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position,

        COUNT(*) AS observed_days,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM content_month

WHERE impressions > 0
  AND ctr_pct IS NOT NULL
  AND avg_position IS NOT NULL
  AND avg_position > 0
""").df()

print(f"Usable March content items: {len(march):,}")

display(march.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Usable March content items: 175,304


,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,observed_days,first_report_date,last_report_date
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.175439,4.394234,31,2026-03-01,2026-03-31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,7.842593,26,2026-03-01,2026-03-31
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,8.454069,30,2026-03-01,2026-03-31
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.422238,6.320337,31,2026-03-01,2026-03-31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.577617,4.459107,31,2026-03-01,2026-03-31


In [6]:
print("Rows:", f"{len(march):,}")
print("Clients:", f"{march['client_hash_id'].nunique():,}")
print("Content items:", f"{march['content_hash_id'].nunique():,}")

print("\nDate range:")
print(march["first_report_date"].min(), "to", march["last_report_date"].max())

print("\nCTR summary:")
display(march["ctr_pct"].describe())

print("\nImpression summary:")
display(march["impressions"].describe())

Rows: 175,304
Clients: 47
Content items: 175,304

Date range:
2026-03-01 00:00:00 to 2026-03-31 00:00:00

CTR summary:


,ctr_pct
count,175304.000000
mean,0.436343
std,3.451047
min,0.000000
25%,0.000000
50%,0.000000
75%,0.218341
max,100.000000



Impression summary:


,impressions
count,175304.000000
mean,1600.961946
std,5451.604207
min,1.000000
25%,21.000000
50%,178.000000
75%,1053.000000
max,617124.000000


### Signal 1 — CTR vs. average position

This is the flag-linked signal.

FlyRank's session connected CTR-vs-position patterns with CTR-fix logic. I will check whether CTR differs meaningfully across position buckets.

**Why it matters for the rule:** a low CTR is more interesting when compared with pages receiving similar search positions, rather than comparing every page to one global CTR threshold.

The verdict will be based on the observed March bucket table.

In [7]:
position_data = march[
    (march["impressions"] > 0) &
    (march["avg_position"].notna()) &
    (march["avg_position"] > 0)
].copy()

position_data["position_bucket"] = pd.cut(
    position_data["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=[
        "1-3",
        "4-10",
        "11-20",
        "21-50",
        "51+"
    ]
)

position_summary = (
    position_data
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr_pct=("ctr_pct", "median"),
        median_position=("avg_position", "median"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

display(position_summary)

,position_bucket,n,median_ctr_pct,median_position,median_impressions
0,1-3,13136,0.096246,2.278532,654.0
1,4-10,81619,0.000000,6.170215,204.0
2,11-20,32548,0.000000,13.954362,256.0
3,21-50,34783,0.000000,29.847951,165.0
4,51+,13218,0.000000,64.964136,45.0


### Signal 1 verdict: CONFIRMED

The position bucket table is used to establish that CTR should be interpreted relative to search position rather than using one global CTR threshold.

This is an observed directional relationship in the March data. It does not prove that improving position or changing a page will cause CTR to increase.

### Signal 2 — Search impression volume

The second signal is search impression volume.

I expect pages with higher impression volume to be more useful for a prioritization queue because a CTR improvement on a high-visibility page potentially affects more observed search impressions.

This is a prioritization assumption rather than a causal claim.

In [8]:
volume_data = march[
    march["impressions"] > 0
].copy()

volume_data["volume_bucket"] = pd.qcut(
    volume_data["impressions"],
    q=4,
    labels=[
        "Q1 - Lowest",
        "Q2",
        "Q3",
        "Q4 - Highest"
    ],
    duplicates="drop"
)

volume_summary = (
    volume_data
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median"),
        median_ctr_pct=("ctr_pct", "median")
    )
    .reset_index()
)

display(volume_summary)

,volume_bucket,n,median_impressions,median_ctr_pct
0,Q1 - Lowest,44350,5.0,0.000000
1,Q2,43321,69.0,0.000000
2,Q3,43810,427.0,0.000000
3,Q4 - Highest,43823,3042.0,0.194951


### Signal 2 verdict: CONFIRMED

The volume buckets provide a simple way to distinguish pages with relatively little observed search visibility from pages with much larger impression volume.

I use impression volume as a prioritization signal rather than claiming that high impressions automatically mean that a page can be improved.

## 2. Build the ranked queue

The rule has two conditions:

1. The page must have at least the median impression volume in the March dataset.
2. The page's CTR must be below the median CTR for its average-position bucket.

For qualifying pages, the score combines:

- impression volume, and
- the size of the CTR gap below the position-bucket median.

A larger score means more observed impressions combined with a larger relative CTR opportunity.

Only qualifying pages receive the reason code `HIGH_VOLUME_LOW_CTR` and action `REVIEW_CTR`.

In [9]:
position_medians = (
    position_data
    .groupby("position_bucket", observed=False)["ctr_pct"]
    .median()
    .rename("position_bucket_median_ctr")
    .reset_index()
)

position_data = position_data.merge(
    position_medians,
    on="position_bucket",
    how="left"
)

position_data["ctr_opportunity"] = (
    position_data["ctr_pct"]
    < position_data["position_bucket_median_ctr"]
)

ctr_summary = (
    position_data
    .groupby("ctr_opportunity")
    .agg(
        n=("content_hash_id", "size"),
        median_ctr_pct=("ctr_pct", "median"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

display(ctr_summary)

,ctr_opportunity,n,median_ctr_pct,median_impressions
0,False,168737,0.0,189.0
1,True,6567,0.0,4.0


In [10]:
# Median impression threshold.
impression_threshold = march["impressions"].median()

print(
    "Median March impressions threshold:",
    f"{impression_threshold:,.2f}"
)

queue = position_data.copy()

queue["high_volume"] = (
    queue["impressions"] >= impression_threshold
)

queue["qualifies"] = (
    queue["high_volume"] &
    queue["ctr_opportunity"]
)

# CTR gap below the typical CTR for the position bucket.
queue["ctr_gap_pct"] = (
    queue["position_bucket_median_ctr"]
    - queue["ctr_pct"]
).clip(lower=0)

# Opportunity score:
# more impressions + larger CTR gap = higher priority.
queue["score"] = (
    queue["impressions"]
    * queue["ctr_gap_pct"]
)

# Keep only pages that meet the baseline rule.
queue = queue[
    queue["qualifies"]
].copy()

queue["reason_code"] = "HIGH_VOLUME_LOW_CTR"
queue["action"] = "REVIEW_CTR"

queue = queue.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

print(f"Qualifying pages: {len(queue):,}")

display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr_pct",
            "avg_position",
            "position_bucket",
            "position_bucket_median_ctr",
            "ctr_gap_pct",
            "score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Median March impressions threshold: 178.00
Qualifying pages: 1,856


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,position_bucket,position_bucket_median_ctr,ctr_gap_pct,score,reason_code,action
0,1,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,0.043306,1.488604,1-3,0.096246,0.052941,4278.729548,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1,2,client_62f4a7e64f5e0096,content_fc67675904376267,60172.0,18.0,0.029914,2.261303,1-3,0.096246,0.066332,3991.337825,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
2,3,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000.0,1.0,0.002632,2.740744,1-3,0.096246,0.093615,3557.362849,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
3,4,client_e547b89c05043229,content_c46df0fa61530d86,70398.0,42.0,0.059661,1.556258,1-3,0.096246,0.036586,2575.553417,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
4,5,client_fef1a8f436438636,content_66bf45eb0c5bb550,24259.0,1.0,0.004122,2.784944,1-3,0.096246,0.092124,2234.841193,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
5,6,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,25941.0,3.0,0.011565,2.418270,1-3,0.096246,0.084682,2196.727623,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
6,7,client_73cda7b4e4f265ea,content_1d7764b642f7bb9f,23402.0,4.0,0.017093,1.843455,1-3,0.096246,0.079154,1852.358037,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
7,8,client_62f4a7e64f5e0096,content_805fd45594ea2398,30792.0,12.0,0.038971,2.920553,1-3,0.096246,0.057275,1763.618864,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
8,9,client_fef1a8f436438636,content_8028c5009353ea97,27747.0,13.0,0.046852,2.994262,1-3,0.096246,0.049394,1370.548604,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
9,10,client_e547b89c05043229,content_04fb6296acff6360,28737.0,14.0,0.048718,2.635650,1-3,0.096246,0.047529,1365.832531,HIGH_VOLUME_LOW_CTR,REVIEW_CTR


In [11]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir /
    "baseline_action_score.csv"
)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position",
    "position_bucket",
    "position_bucket_median_ctr",
    "ctr_gap_pct",
    "score",
    "reason_code",
    "action"
]

queue[output_columns].to_csv(
    output_file,
    index=False
)

print(f"Saved: {output_file}")
print(f"Rows written: {len(queue):,}")

Saved: work/outputs/baseline_action_score.csv
Rows written: 1,856


In [12]:
saved = pd.read_csv(output_file)

print("Saved rows:", f"{len(saved):,}")
print("Saved columns:")
print(saved.columns.tolist())

display(saved.head(10))

Saved rows: 1,856
Saved columns:
['rank', 'client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'ctr_pct', 'avg_position', 'position_bucket', 'position_bucket_median_ctr', 'ctr_gap_pct', 'score', 'reason_code', 'action']


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,position_bucket,position_bucket_median_ctr,ctr_gap_pct,score,reason_code,action
0,1,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,0.043306,1.488604,1-3,0.096246,0.052941,4278.729548,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1,2,client_62f4a7e64f5e0096,content_fc67675904376267,60172.0,18.0,0.029914,2.261303,1-3,0.096246,0.066332,3991.337825,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
2,3,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000.0,1.0,0.002632,2.740744,1-3,0.096246,0.093615,3557.362849,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
3,4,client_e547b89c05043229,content_c46df0fa61530d86,70398.0,42.0,0.059661,1.556258,1-3,0.096246,0.036586,2575.553417,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
4,5,client_fef1a8f436438636,content_66bf45eb0c5bb550,24259.0,1.0,0.004122,2.784944,1-3,0.096246,0.092124,2234.841193,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
5,6,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,25941.0,3.0,0.011565,2.418270,1-3,0.096246,0.084682,2196.727623,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
6,7,client_73cda7b4e4f265ea,content_1d7764b642f7bb9f,23402.0,4.0,0.017093,1.843455,1-3,0.096246,0.079154,1852.358037,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
7,8,client_62f4a7e64f5e0096,content_805fd45594ea2398,30792.0,12.0,0.038971,2.920553,1-3,0.096246,0.057275,1763.618864,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
8,9,client_fef1a8f436438636,content_8028c5009353ea97,27747.0,13.0,0.046852,2.994262,1-3,0.096246,0.049394,1370.548604,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
9,10,client_e547b89c05043229,content_04fb6296acff6360,28737.0,14.0,0.048718,2.635650,1-3,0.096246,0.047529,1365.832531,HIGH_VOLUME_LOW_CTR,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The following review treats the ranked queue as a decision-support list rather than a list of guaranteed opportunities.

For each item I record the action, the reason it received a high score, a confidence note, and what could make the recommendation wrong.

In [13]:
top20 = queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda r: (
        "Higher confidence in the baseline signal because the page "
        "has at least median impression volume and CTR below the "
        "median for its position bucket."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "Low CTR may be appropriate for the query intent, SERP features, "
    "brand terms, or the page's actual search position; impression "
    "volume alone does not prove an optimization opportunity."
)

review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(review)

,rank,content_hash_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_306bc78dff1eb683,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,4278.729548,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
1,2,content_fc67675904376267,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,3991.337825,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
2,3,content_d61fc394d10cba41,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,3557.362849,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
3,4,content_c46df0fa61530d86,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,2575.553417,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
4,5,content_66bf45eb0c5bb550,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,2234.841193,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
5,6,content_b9acd1ebff7d34ff,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,2196.727623,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
6,7,content_1d7764b642f7bb9f,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,1852.358037,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
7,8,content_805fd45594ea2398,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,1763.618864,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
8,9,content_8028c5009353ea97,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,1370.548604,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...
9,10,content_04fb6296acff6360,REVIEW_CTR,HIGH_VOLUME_LOW_CTR,1365.832531,Higher confidence in the baseline signal becau...,Low CTR may be appropriate for the query inten...


## 4. Weak picks + leakage check

### Weak picks

The weakest part of this baseline is that it does not know:

- search intent,
- SERP features,
- brand strength,
- whether the current title and snippet are already appropriate,
- whether the page is commercially important,
- or whether the observed CTR difference is caused by something outside the page.

Therefore, a high score is only a reason to review the page, not a recommendation to make a guaranteed change.

### Leakage check

The baseline uses March 2026 search observations only.

I deliberately exclude:

- `trend_direction`
- `trend_pct`
- `is_declining_label`

These fields are label-derived and therefore must not be used as model or baseline features.

No future outcome window is used to construct the score.

In [14]:
# Show some qualifying items with relatively small scores.
# These are useful examples of cases that could be weak recommendations.

weak_picks = (
    queue
    .sort_values(
        ["score", "impressions"],
        ascending=[True, True]
    )
    .head(10)
)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "impressions",
            "ctr_pct",
            "avg_position",
            "position_bucket_median_ctr",
            "ctr_gap_pct",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

,rank,content_hash_id,impressions,ctr_pct,avg_position,position_bucket_median_ctr,ctr_gap_pct,score,reason_code,action
1855,1856,content_da44d5bf2e75f6a3,1042.0,0.095969,1.215558,0.096246,0.000277,0.288739,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1854,1855,content_13dd3198c288e40f,1045.0,0.095694,1.147766,0.096246,0.000553,0.577478,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1853,1854,content_0ce167e0fb674bcd,3124.0,0.096031,1.209743,0.096246,0.000216,0.673725,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1852,1853,content_801b96ca58eaa8de,1048.0,0.095420,2.088292,0.096246,0.000827,0.866218,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1851,1852,content_3cc063b3e8e38957,2089.0,0.095740,2.111869,0.096246,0.000507,1.058710,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1850,1851,content_2668936c12aea527,1050.0,0.095238,2.812503,0.096246,0.001008,1.058710,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1849,1850,content_f035c7e3ec91ecbb,4168.0,0.095969,2.958531,0.096246,0.000277,1.154957,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1847,1848,content_6d39e678ecda2135,2090.0,0.095694,2.566243,0.096246,0.000553,1.154957,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1848,1849,content_2b5242bc35fb162c,2090.0,0.095694,2.392073,0.096246,0.000553,1.154957,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1845,1846,content_bf2c1278879f0de7,1051.0,0.095147,1.864227,0.096246,0.001099,1.154957,HIGH_VOLUME_LOW_CTR,REVIEW_CTR


In [15]:
# Columns that must never be used as baseline inputs.
forbidden = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

# These are the actual columns used by the rule.
used_columns = {
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position",
    "position_bucket",
    "position_bucket_median_ctr",
    "ctr_gap_pct"
}

leakage_columns_used = forbidden.intersection(
    used_columns
)

print(
    "Forbidden label-derived columns used:",
    leakage_columns_used
)

assert len(leakage_columns_used) == 0

print("Leakage check: PASSED")

Forbidden label-derived columns used: set()
Leakage check: PASSED


In [16]:
# Confirm that this baseline only uses March 2026 observations.

print(
    "First observed date:",
    march["first_report_date"].min()
)

print(
    "Last observed date:",
    march["last_report_date"].max()
)

print(
    "\nThe baseline uses March 2026 observations only."
)

assert (
    march["last_report_date"].max().month == 3
)

assert (
    march["last_report_date"].max().year == 2026
)

print("Future-window check: PASSED")

First observed date: 2026-03-01 00:00:00
Last observed date: 2026-03-31 00:00:00

The baseline uses March 2026 observations only.
Future-window check: PASSED


## Baseline conclusion

The baseline produces a ranked action queue for content pages with:

1. relatively high search impression volume, and
2. CTR below the median for pages in a similar search-position bucket.

The queue is intended to help an editor decide which pages to review first for possible CTR improvements.

The two signals are observable in the decision-period data, and the baseline does not use label-derived trend fields or future outcomes.

The main limitation is that low CTR does not necessarily mean that a page should be changed. Search intent, SERP features, brand terms, and other contextual factors are not represented in this simple rule.

The Week-5 model should therefore be evaluated against this baseline rather than assuming that the baseline recommendations are correct.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.